# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

H. Zhang  ->  H. Zhang  |  ['H. Zhang']
T. Henning  ->  T. Henning  |  ['T. Henning']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
M. Benisty  ->  M. Benisty  |  ['M. Benisty']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
H. Jiang  ->  H. Jiang  |  ['H. Jiang']
S. Shahaf  ->  S. Shahaf  |  ['S. Shahaf']
H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']


K. Kreckel  ->  K. Kreckel  |  ['K. Kreckel']
Arxiv has 84 new papers today
          9 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/9 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.07006
extracting tarball to tmp_2611.07006... done.
Retrieving document from  https://arxiv.org/e-print/2611.07034


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.07006/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'authors' from 'tmp_2611.07006/authors.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


extracting tarball to tmp_2611.07034...

 done.


/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.07034/main.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'big_table' from 'tmp_2611.07034/big_table.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'sideris' from 'tmp_2611.07034/sideris.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.18/x64/lib/python3.11/site-packages/arxiv_

... (14 outputs left out)

S. Shahaf  ->  S. Shahaf  |  ['S. Shahaf']
H.-W. Rix  ->  H.-W. Rix  |  ['H.-W. Rix']


Retrieving document from  https://arxiv.org/e-print/2611.08336


extracting tarball to tmp_2611.08336...

 done.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07034-b31b1b.svg)](https://arxiv.org/abs/2611.07034) | **The JWST Early Release Science Program for Direct Observations of Exoplanetary Systems IX: A Multi-Wavelength Framework for Source Characterization in JWST Coronagraphic Imaging of the HIP 65426 System**  |
|| E. L. Bruinsma, et al. -- incl., <mark>T. Henning</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *Accepted for publication in ApJL, 26 pages*|
|**Abstract**|            We present an updated analysis of the JWST coronagraphic imaging data of the HIP 65426 system. We aim to search for and characterize potential planetary companions as well as provide updated astrometry and photometry for the known exoplanet HIP 65426 b following updates to the JWST and spaceKLIP pipelines. Our measured photometry and astrometry show significant differences to those in Carter et al. (2023) but are consistent to within $2.2{\sigma}$ in separation, $1.5{\sigma}$ in position angle, and $3.7{\sigma}$ in flux. Our results indicate that pipeline updates have changed the photometry and astrometry significantly in some filters, especially in terms of observed flux. We see the greatest changes in separation in F444W and F1550C, position angle in F250M, and flux in F250M, F300M, F410M, F1140C, and F1550C. The SED of HIP 65426 b remains consistent with a young, giant planet best fit by models with a cloudy atmosphere and an effective temperature estimate of ~1436 K. Using both NIRCam and MIRI data, we identified 21 candidate companion sources in addition to the known planet. Of the 21 sources, 13 were attributed to astrophysical sources with eight background stars, four galaxies, one unknown, and zero planets. While one of these sources was detected in Chauvin et al. (2017), the remaining point sources were undetected in previous ground-based observations, which makes this work an initial epoch of astrometry for these objects. One galaxy in particular appears as a point source in MIRI but is highly extended in NIRCam and exposes a potential drawback to single-epoch MIRI imaging in a blind search for companions.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07253-b31b1b.svg)](https://arxiv.org/abs/2611.07253) | **Coordinated Space and Ground Monitoring of Accretion Bursts in a Protoplanetary Disk: MIRI and iSHELL observe the long-term modulation of gas within a small inner disk cavity**  |
|| J. Hyden, et al. -- incl., <mark>M. Benisty</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *Accepted for publication on ApJ*|
|**Abstract**|            DQ Tau is a TTauri eccentric binary with pulsed accretion bursts regulated by its $\sim 16$-day orbital period. We have monitored this system with seven instruments from ground and space over 40 days. We present results on molecular gas variability from nine JWST-MIRI and nine IRTF-iSHELL epochs over three sequential accretion bursts with accretion luminosity pulses increasing by factors of $\sim 9$-80. A fundamental new measurement from this analysis is the line broadening of CO (from iSHELL) and H$_2$O (from MIRI) lines, which we use to estimate Keplerian radii and their variability in parallel to the line luminosity. This unprecedented dataset clearly shows that the molecular emission does not react like HI and the 5-$\mu$m continuum with each accretion burst: the line luminosity and width of all hot molecular gas tracers decrease over $\sim30$-35 days, consistent with an emitting region decreasing in temperature and mass as it recedes to larger radii within the $\sim 0.2$-0.4 au inner cavity perturbed by the binary. The infrared molecular spectrum of DQ Tau is affected more by disk dynamics then by chemical changes triggered by variable accretion, supporting model expectations for chemical relaxation timescales that are much longer than the duration of these accretion bursts ($\gg 10$ days). The variability observed in all tracers in this study is consistent with a scenario where gas mass builds up near the disk edge to modulate the burst peak intensity and gas density in the inner cavity every $\sim4$-5 orbital periods. The origin of this modulation is still unclear.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08282-b31b1b.svg)](https://arxiv.org/abs/2611.08282) | **Mitigating scanning-induced systematics in Hipparcos and implications for Gaia astrometry**  |
|| A. Binnenfeld, et al. -- incl., <mark>S. Shahaf</mark>, <mark>H.-W. Rix</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *6 pages, 5 figures. Accepted for publication in Astronomy & Astrophysics*|
|**Abstract**|            Space-based astrometric missions such as Hipparcos and Gaia rely on one-dimensional scanning strategies that couple the measured signals with both observation times and scanning geometry. This coupling can introduce spurious periodicities and complicate their interpretation. We present a non-parametric extension of the phase-distance correlation (PDC) periodogram that quantifies and reduces dependence on scan geometry at the period-search stage. Using Hipparcos data, we identified recurrent periodicities associated with characteristic scanning-law timescales and showed that they are strongly suppressed by the scan-angle-controlled statistic. We then applied the method to Gaia Data Release 4 (DR4) prerelease epoch astrometry, where a recurrent ~54-day feature is suppressed across multiple sources while independently known orbital signals remain detectable. The method provides a diagnostic of scan-angle dependence and a way to mitigate its impact on period searches. These results are particularly relevant for Gaia DR4, where public epoch astrometry will enable large-scale searches for companions ranging from exoplanets to dormant black holes.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.08336-b31b1b.svg)](https://arxiv.org/abs/2611.08336) | **SDSS-V LVM: Spatial Resolution Effects on Diagnostic Diagrams in the Magellanic Clouds**  |
|| A. Z. Lugo-Aranda, et al. -- incl., <mark>K. Kreckel</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *20 pages, 9 figures*|
|**Abstract**|            We investigate how spatial resolution affects emission-line diagnostics in the interstellar medium (ISM) of the Small and Large Magellanic Clouds (SMC and LMC) using SDSS Local Volume Mapper (LVM) data obtained at and degraded to 19, 42, 170 pc (for the SMC) and 17, 68, 272 pc (for the LMC), respectively. We compare the distributions of strong emission-line ratios through 1D and 2D Kolmogorov Smirnov tests. We deduced that diagnostics involving ionization and density sensitive ratios such as [OIII]{\lambda}5007/H\b{eta}, [SII]{\lambda}{\lambda}6717+6731/H{\alpha} and [NII]{\lambda}6584/H{\alpha} are the most strongly impacted by resolution loss in this order. The SMC shows significant deviations even from intermediate scales (42 pc), whereas for the LMC, the coarsest resolution (272 pc) remains relatively stable, indicating that the SMC is intrinsically more affected by spatial mixing. Resolution degradation produces a marked homogenization effect: extreme values present at the finest scales are progressively diluted, causing diagnostic diagrams to converge toward their mean loci. Diffuse ionized gas is particularly sensitive to this mixing, while H{\alpha} selected ionized regions exhibit more robust behavior across resolutions. The sensitivity of diagnostic diagrams involving emission-line ratios, equivalent widths, and kinematic properties varies with both spatial resolution and galaxy, with the SMC generally showing stronger changes than the LMC. Our results demonstrate that sub-50 pc resolution is required to resolve the detailed structure of low-metallicity galaxies, and they highlight at least the need for careful interpretation of line-ratio diagnostics in extragalactic studies where coarse spatial resolution is unavoidable.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: '69117' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07006-b31b1b.svg)](https://arxiv.org/abs/2611.07006) | **Black Hole Seeds and Little Red Dots: Connecting Formation and Growth with a JWST Multi-Cycle Treasury Program**  |
|| F. Pacucci, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *Theory white paper on black hole seeds and Little Red Dots, submitted to the STScI call for JWST Multi-Cycle Treasury concept white papers as part of the Genesis Initiative (this http URL). 5 pages (plus affiliations and references)*|
|**Abstract**|            JWST has revealed an abundant population of faint active galactic nuclei and a new class of compact, high-redshift sources known as Little Red Dots (LRDs), whose physical nature remains debated, with accreting massive black holes among the leading explanations. The emerging demographics of early black holes, with their high inferred number densities and, in some systems, the apparent excess of black hole mass relative to stellar mass, challenge simple scenarios of black hole seeding and growth. Despite these observational breakthroughs, the origin of early massive black holes remains uncertain, because "black holes have no memory": current samples cannot separate the imprint of seeding from subsequent growth. This white paper identifies the multi-cycle JWST observations needed to revolutionize our understanding of early black hole growth by: (i) distinguishing seeding pathways, (ii) constraining early black hole growth mechanisms, (iii) resolving the physical nature of broad-line emitters and LRDs, and (iv) testing black hole-galaxy co-evolution. These tasks require uniformly selected samples extending to low black hole and host-galaxy masses, allowing self-consistent measurements of black hole mass functions, Eddington-ratio distributions, and host properties. Grating spectroscopy, deep MIRI imaging, and variability analysis through repeated observations of the same fields would provide complementary tests of black hole formation and growth, their connection to host-galaxy evolution, and the nature of newly discovered populations of LRDs. In this white paper, we argue that these joint tests require a coordinated JWST Multi-Cycle Treasury program; such a program could provide the depth, uniformity, and time baseline needed to place substantially stronger, complementary constraints on early black hole formation, growth, and their connection to galaxy evolution.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07237-b31b1b.svg)](https://arxiv.org/abs/2611.07237) | **Dust Survival in Lensed Quiescent Galaxies at $z>1$ Depends on How They Quench**  |
|| S. Bodansky, et al. -- incl., <mark>Y. Zhang</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *Submitted to ApJ. 20 pages, 11 figures*|
|**Abstract**|            The interstellar medium of massive quiescent galaxies holds key clues to how they quench, yet remain poorly understood. Although dust is expected to be depleted rapidly ($\lesssim$100 Myr) after quenching, some quiescent galaxies host substantial dust reservoirs, raising the question of whether dust survives quenching or is replenished afterward. We present ALMA Band 6 continuum observations ($\lambda_{\mathrm{rest}}$$>$250$\mu$m) of eight strongly-lensed, spectroscopically-selected quiescent galaxies at z$\sim$1-2.5 from the COOL-LAMPS and REQUIEM surveys, plus five archival REQUIEM systems. The sample spans more than two orders of magnitude in dust-to-stellar-mass ratio ($f_{\mathrm{dust}}<10^{-5}$ to $>10^{-3}$), and two galaxies host extended, resolved dust. Comparing $f_{\mathrm{dust}}$ with time since quenching, roughly half the sample, preferentially at higher redshift, is consistent with rapid quenching and harsh dust destruction, while lower-redshift, more slowly quenched systems are consistent with milder destruction, possibly reflecting an evolving dominant quenching mode. The two most dust-rich galaxies exceed even the mildest model destruction tracks, with J1201-3952 retaining the highest $f_{\mathrm{dust}}$ $\sim$1.8 Gyr after quenching. If these systems are truly quiescent, their dust masses approach maximum stellar yields, pointing to ISM grain regrowth and/or TP-AGB dust production. However due to our single-band continuum data, $f_{\mathrm{dust}}$ is uncertain ($\pm$0.3 dex systematic uncertainty from the unknown dust temperature) and we cannot rule out obscured star formation in these two galaxies. Our results suggest that dust survival depends on how galaxies quench: rapid, harsh quenching efficiently destroys dust, while slower, milder quenching can leave dust reservoirs that persist or are replenished long after star formation ceases.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07427-b31b1b.svg)](https://arxiv.org/abs/2611.07427) | **Prospects for detecting hot-Jupiter tidal decay through transit timing with the mission Earth 2.0**  |
|| W. Wang, et al. -- incl., <mark>Y. Wang</mark>, <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *11 pages, 6 figures*|
|**Abstract**|            Orbital decay of hot Jupiters provides a unique opportunity to probe tidal dissipation in star-planet systems. However, despite extensive observational efforts, only one hot Jupiter, namely WASP-12b, currently exhibits conclusive observational evidence for orbital decay. Expanding this sample is therefore crucial for establishing a statistical understanding of tidal interactions in planetary systems. Long-term transit timing variations (TTVs) provide a direct observational probe of orbital decay induced by tidal dissipation between a hot Jupiter and its host star. The Earth 2.0 (ET) mission will revisit the original Kepler field, delivering high-precision transit timing measurements and extending the observational baseline to approximately 24 years (2009--2033). In this work, we aim to assess the capability of ET to detect orbital decay in hot Jupiters previously observed by Kepler. We generate four years of simulated ET light curves based on the expected photometric precision and observing cadence of the mission and analyze them jointly with archival Kepler and TESS observations. Using these combined datasets, we perform long-term TTV analyses and predict the achievable precision for measuring orbital period variations. We find that ET can substantially improve the precision of orbital decay measurements in hot Jupiter systems by a factor of 7.2. Assuming the stellar tidal quality factor inferred from previous observations, ET is expected to detect orbital decay in three additional hot Jupiter systems. The expanded sample of hot Jupiters with measured orbital decay will establish ET as a powerful probe of tidal dissipation in star-planet systems, providing tighter constraints on stellar tidal quality factors and new insights into the tidal evolution of hot Jupiters.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07433-b31b1b.svg)](https://arxiv.org/abs/2611.07433) | **MP-DualNet: A Dual-Branch Deep Learning Approach for Magnetopause Boundary Detection Using Simulated SXI Images**  |
|| C. Zhong, et al. -- incl., <mark>Y. Wang</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| **|
|**Abstract**|            The Earth's magnetopause is a key boundary governing solar wind-magnetosphere coupling. The Solar wind Magnetosphere Ionosphere Link Explorer (SMILE) mission will provide global soft X-ray imaging of the magnetosphere through solar wind charge exchange (SWCX) emission, offering a new opportunity for remote sensing of magnetopause dynamics. However, the limited field of view (FOV) and variable observation geometry of the Soft X-ray Imager (SXI) pose challenges for reliable magnetopause detection. In this study, we develop MP-DualNet, an end-to-end dual-branch deep learning framework for automated magnetopause detection from simulated SXI observations. The framework jointly performs coarse localization to determine magnetopause visibility and relative position within the FOV, and fine-scale boundary localization through pixel-level segmentation. The model is trained and evaluated using line-of-sight magnetopause envelope maps derived from global hybrid simulations under different solar wind and interplanetary magnetic field conditions. The results demonstrate stable and consistent performance, with a classification accuracy exceeding 0.94 and a segmentation IoU above 0.97. For the test set, the mean absolute angular displacement from the reference boundary was 0.488 degrees. The model maintains robust performance across a wide range of noise levels and generalizes well, as demonstrated by its application to an independent storm event. These results indicate that MP-DualNet provides a reliable and generalizable approach for magnetopause detection in SXI observations, supporting future 3D reconstruction and space weather studies in the context of the SMILE mission.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.07832-b31b1b.svg)](https://arxiv.org/abs/2611.07832) | **Design and Commissioning of the BURSTT-256 Main Antenna Array for the Fast Radio Burst Survey in the Local Universe**  |
|| S.-H. Wang, et al. -- incl., <mark>H. Jiang</mark> |
|*Appeared on*| *2026-11-06*|
|*Comments*| *22 pages, 14 figures; submitted to PASP*|
|**Abstract**|            The Bustling Universe Radio Survey Telescope in Taiwan (BURSTT) is a radio interferometer array designed for surveying fast radio bursts (FRBs) in the nearby Universe with a wide instantaneous field of view (FoV) of $\approx$6000 deg$^2$ and precisely localizing them with sub-arcsecond angular resolution. The first phase of BURSTT, BURSTT-256, consists of a main antenna array comprising 256 log-periodic dipole antennas (LPDAs) observing at 400-800 MHz for real-time beamforming and FRB search, currently with 256 beams covering $\approx$ 3200 deg$^2$ FoV, together with several smaller outrigger stations with 16-64 antennas and separated by $\mathcal{O}(10^2 - 10^3)$ km baselines for joint observations to localize FRBs using very-long baseline interferometry. This work presents the design, deployment, and commissioning of the BURSTT-256 main array, including site selection, radio-frequency (RF) background characterization, receiver antenna array and RF front-end, as well as real-time, factorized digital beamforming backend that currently generates 256 beams. The system's on-sky performance, sensitivity, and beamforming fidelity have been calibrated through observations of bright celestial calibrators. The antenna array in the initial phase using commercial LPDAs has a system equivalent flux density of $\approx$11 kJy, and is expected to achieve $\approx$4 kJy using custom-design LPDAs after optimization in late 2026. BURSTT-256 serves as a critical pathfinder for future expansion of the array to 2048 elements and beyond, operating at 300-700 MHz with $\approx$6000 deg$^2$ FoV. It establishes the framework required to discover and localize nearby FRBs with sub-arcsecond precision. By substantially increasing the number of well-localized nearby FRBs, BURSTT will facilitate multi-wavelength and multi-messenger counterpart identification, helping to unveil the elusive origin of FRBs.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.07034/./b.png', 'tmp_2611.07034/./c1.png', 'tmp_2611.07034/./c20.png', 'tmp_2611.07034/./c2.png', 'tmp_2611.07034/./c4.png', 'tmp_2611.07034/./c6.png', 'tmp_2611.07034/./c9.png', 'tmp_2611.07034/./c11.png', 'tmp_2611.07034/./c3.png', 'tmp_2611.07034/./c5.png', 'tmp_2611.07034/./c8.png', 'tmp_2611.07034/./c10.png', 'tmp_2611.07034/./c12.png', 'tmp_2611.07034/./sources_f444_f1550.png']
copying  tmp_2611.07034/./b.png to _build/html/
copying  tmp_2611.07034/./c1.png to _build/html/
copying  tmp_2611.07034/./c20.png to _build/html/
copying  tmp_2611.07034/./c2.png to _build/html/
copying  tmp_2611.07034/./c4.png to _build/html/
copying  tmp_2611.07034/./c6.png to _build/html/
copying  tmp_2611.07034/./c9.png to _build/html/
copying  tmp_2611.07034/./c11.png to _build/html/
copying  tmp_2611.07034/./c3.png to _build/html/
copying  tmp_2611.07034/./c5.png to _build/html/
copying  tmp_2611.07034/./c8.png to _build/html/
copying  tmp_2611.07034/./c10.png to _build/html

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\todo}[1]{\textcolor{red}{[#1]}}$
$\newcommand{\um}{\text{ \textmu m\xspace}}$
$\newcommand{\pcc}{ cm^{-3}}$
$\newcommand{\teff}{\text{T}_\mathrm{eff}}$
$\newcommand{\logg}{log (\text{g})}$
$\newcommand{\lbol}{\text{L}_\mathrm{bol}}$
$\newcommand{\mjup}{\text{M}_\mathrm{J}}$
$\newcommand{\msun}{\text{M}_\mathrm{\odot}}$
$\newcommand{\rjup}{\text{R}_\mathrm{J}}$
$\newcommand{\rsun}{\text{R}_\mathrm{\odot}}$
$\newcommand{\lsun}{\text{L}_\mathrm{\odot}}$
$\newcommand{\jhu}{Department of Physics \& Astronomy, Johns Hopkins University, 3400 N. Charles Street, Baltimore, MD 21218, USA}$
$\newcommand{\stsci}{Space Telescope Science Institute, 3700 San Martin Dr, Baltimore, MD 21218, USA}$
$\newcommand{\uofm}{Department of Astronomy, University of Michigan, 1085 S. University Ave, Ann Arbor, MI 48109, USA}$
$\newcommand{\IAedinburgh}{Institute for Astronomy, University of Edinburgh, Royal Observatory, Blackford Hill, Edinburgh, EH9 3HJ, UK}$
$\newcommand{\CESedinburgh}{Centre for Exoplanet Science, University of Edinburgh, Edinburgh, EH9 3HJ, UK}$
$\newcommand{\exeter}{Department of Physics and Astronomy, University of Exeter, Stocker Road, Exeter EX4 4QL, UK}$
$\newcommand{\marseille}{Laboratoire d'Astrophysique de Marseille, 38 Rue Fréderic Joliot Curie, 130013 Marseille, France}$
$\newcommand{\toronto}{Department of Astronomy and Astrophysics, University of Toronto, 50 St. George Street, Toronto, ON M5S 3H4, Canada}$
$\newcommand{\mitkavli}{MIT Kavli Institute for Astrophysics and Space Research, Massachusetts Institute of Technology, Cambridge, MA 02139, USA}$
$\newcommand{\UnivDiegoPortales}{Instituto de Estudios Astrofísicos, Facultad de Ingeniería y Ciencias, Universidad Diego Portales, Av. Ejército Libertador 441, Santiago, Chile}$
$\newcommand{\yems}{Millennium Nucleus on Young Exoplanets and their Moons (YEMS)}$
$\newcommand{\padova}{INAF-Osservatorio Astronomico di Padova, Vicolo dell'Osservatorio 5, Padova, Italy, 35122-I}$
$\newcommand{\ucsd}{Department of Astronomy \& Astrophysics, University of California San Diego, La Jolla, CA 92093, USA}$
$\newcommand{\AstrobioCenter}{Astrobiology Center, 2-21-1 Osawa, Mitaka, Tokyo 181-8588, Japan}$
$\newcommand{\mpia}{Max Planck Institute for Astronomy, Königstuhl 17, 69117 Heidelberg, Germany}$
$\newcommand{\stockholm}{Department of Astronomy, Stockholm University, Stockholm, SE-10691, Sweden}$
$\newcommand{\cambridge}{Institute of Astronomy, University of Cambridge, Madingley Road, Cambridge, CB3 0HA, UK}$
$\newcommand{\amnh}{Department of Astrophysics, American Museum of Natural History, Central Park West at 79th Street, NY 10024, USA}$
$\newcommand{\umd}{Center for Space Sciences and Technology, University of Maryland, Baltimore County, 1000 Hilltop Circle, Baltimore, MD 21250, USA}$
$\newcommand{\gsfcAstro}{Astrophysics Science Division, NASA-GSFC, 8800 Greenbelt Rd, Greenbelt, MD 20771, USA}$
$\newcommand{\gsfcCresst}{Center for Research and Exploration in Space Science and Technology, NASA-GSFC, 8800 Greenbelt Rd, Greenbelt, MD 20771, USA}$
$\newcommand{\CentroAstrobio}{Centro de Astrobiología (CAB, CSIC-INTA), Camino Bajo del Castillo s/n, 28692 Villanueva de la Cañada, Madrid, Spain}$
$\newcommand{\sfsu}{Department of Physics and Astronomy, San Francisco State University, 1600 Holloway Ave., San Francisco, CA 94132, USA}$
$\newcommand{\esa}{European Space Agency (ESA), ESA Office, Space Telescope Science Institute, 3700 San Martin Drive, Baltimore, MD 21218, USA}$
$\newcommand{\ucsc}{Department of Astronomy \& Astrophysics, University of California, Santa Cruz, 1156 High Street, Santa Cruz, CA 95064}$
$\newcommand{\ciera}{Center for Interdisciplinary Exploration and Research in Astrophysics (CIERA), Northwestern University, 1800 Sherman Avenue, Evanston, IL 60201, USA}$
$\newcommand{\uvic}{Department of Physics and Astronomy University of Victoria 3800 Finnerty Road Elliott Building, Room 207. Victoria BC V8P 5C2, Canada}$
$\newcommand{\thefigure}{A.\arabic{figure}}$
$\newcommand{\thetable}{A.\arabic{table}}$
$\newcommand{\thefigure}{A.\arabic{figure}}$
$\newcommand{\theHfigure}{A.\arabic{figure}}$
$\newcommand{\thetable}{A.\arabic{table}}$
$\newcommand{\theHtable}{A.\arabic{table}}$
$\newcommand{\thefigure}{B.\arabic{figure}}$
$\newcommand{\theHfigure}{B.\arabic{figure}}$
$\newcommand{\thetable}{B.\arabic{table}}$
$\newcommand{\theHtable}{B.\arabic{table}}$
$\newcommand{\thefigure}{C.\arabic{figure}}$
$\newcommand{\theHfigure}{C.\arabic{figure}}$
$\newcommand{\thetable}{C.\arabic{table}}$
$\newcommand{\theHtable}{C.\arabic{table}}$
$\newcommand{\thefigure}{D.\arabic{figure}}$
$\newcommand{\theHfigure}{D.\arabic{figure}}$
$\newcommand{\thetable}{D.\arabic{table}}$
$\newcommand{\theHtable}{D.\arabic{table}}$</div>



<div id="title">

# The JWST Early Release Science Program for Direct Observations of Exoplanetary Systems IX: A Multi-Wavelength Framework for Source Characterization in JWST Coronagraphic Imaging of the HIP 65426 System

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.07034-b31b1b.svg)](https://arxiv.org/abs/2611.07034)<mark>Appeared on: 2026-11-06</mark> -  _Accepted for publication in ApJL, 26 pages_

</div>
<div id="authors">

E. L. Bruinsma, et al. -- incl., <mark>T. Henning</mark>

</div>
<div id="abstract">

**Abstract:** We present an updated analysis of the JWST coronagraphic imaging data of the HIP 65426 system. We aim to search for and characterize potential planetary companions as well as provide updated astrometry and photometry for the known exoplanet HIP 65426 b following updates to the JWST and spaceKLIP pipelines. Our measured photometry and astrometry show significant differences to those in [Carter, et. al (2023)](http://dx.doi.org/10.3847/2041-8213/acd93e) but are consistent to within $2.2\sigma$ in separation, $1.5\sigma$ in position angle, and $3.7\sigma$ in flux. Our results indicate that pipeline updates have changed the photometry and astrometry significantly in some filters, especially in terms of observed flux. We see the greatest changes in separation in F444W and F1550C, position angle in F250M, and flux in F250M, F300M, F410M, F1140C, and F1550C. The SED of HIP 65426 b remains consistent with a young, giant planet best fit by models with a cloudy atmosphere and an effective temperature estimate of $\sim$ 1436 K. Using both NIRCam and MIRI data, we identified 21 candidate companion sources in addition to the known planet. Of the 21 sources, 13 were attributed to astrophysical sources with eight background stars, four galaxies, one unknown, and zero planets. While one of these sources was detected in [Chauvin, et. al (2017)](http://dx.doi.org/10.1051/0004-6361/201731152) , the remaining point sources were undetected in previous ground-based observations, which makes this work an initial epoch of astrometry for these objects. One galaxy in particular appears as a point source in MIRI but is highly extended in NIRCam and exposes a potential drawback to single-epoch MIRI imaging in a blind search for companions.

</div>

<div id="div_fig1">

<img src="tmp_2611.07034/./b.png" alt="Fig12.1" width="33%"/><img src="tmp_2611.07034/./c1.png" alt="Fig12.2" width="33%"/><img src="tmp_2611.07034/./c20.png" alt="Fig12.3" width="33%"/>

**Figure 12. -** Example SED fits for each type of object. For each panel, the top is the bandpasses of the photometric data points, the middle is the data plotted alongside the best-fit models, and the bottom is the residuals of each data point relative to the best-fit model. $3\sigma$ upper-limits are shown in grey.
  **Top: **SED of HIP 65426 b, using data from JWST, VLT/SPHERE  ([Chauvin, et. al 2017](http://dx.doi.org/10.1051/0004-6361/201731152)) , VLTI/GRAVITY  ([Blunt, et. al 2023](https://arxiv.org/abs/2310.00148)) , and VLT/NACO  ([Cheetham, et. al 2019](https://ui.adsabs.harvard.edu/abs/2019A&A...622A..80C)) , plotted alongside the best-fit BT-SETTL ($\teff$$=1436^{+13}_{-12}$ K, $\logg$$=3.64\pm0.02$ dex, and $\text{R}_\text{p}=1.43\pm{0.03}$ $\rjup$), Sonora Diamondback ($\teff$$=1210^{+9}_{-7}$ K, $\logg$$=3.76\pm0.06$ dex, and $\text{R}_\text{p}=1.86\pm{0.03}$ $\rjup$), and ATMO  ([Petrus, et. al 2023](https://ui.adsabs.harvard.edu/abs/2023A&A...670L...9P)) ($\teff$$=1376\pm14$ K, $\logg$$=3.87^{+0.07}_{-0.08}$ dex, and $\text{R}_\text{p}=1.50\pm{0.03}$ $\rjup$).
  **Middle: **Example SED of a background star (c1) plotted alongside the best-fit BT-SETTL model, which yields $\teff$$=3298^{+447}_{-572}$ K, $\logg$$=4.5^{+0.64}_{-0.62}$ dex, and $\text{R}=0.10^{+0.007}_{-0.008}$ $\rsun$.
  **Bottom: **Example SED of a background galaxy (c20) plotted alongside the best-fit blackbody and galaxy templates.  (*fig:seds_main*)

</div>
<div id="div_fig2">

<img src="tmp_2611.07034/./c2.png" alt="Fig2.1" width="10%"/><img src="tmp_2611.07034/./c4.png" alt="Fig2.2" width="10%"/><img src="tmp_2611.07034/./c6.png" alt="Fig2.3" width="10%"/><img src="tmp_2611.07034/./c9.png" alt="Fig2.4" width="10%"/><img src="tmp_2611.07034/./c11.png" alt="Fig2.5" width="10%"/><img src="tmp_2611.07034/./c3.png" alt="Fig2.6" width="10%"/><img src="tmp_2611.07034/./c5.png" alt="Fig2.7" width="10%"/><img src="tmp_2611.07034/./c8.png" alt="Fig2.8" width="10%"/><img src="tmp_2611.07034/./c10.png" alt="Fig2.9" width="10%"/><img src="tmp_2611.07034/./c12.png" alt="Fig2.10" width="10%"/>

**Figure 2. -** SED fits for all vetted point-sources not included in Figure \ref{fig:seds_main}. The JWST photometric points are plotted in orange with the best-fit BT-Settl models in black. c9, c10, and c11 also contain the best-fit galaxy SED templates. The residuals for each data point are shown in the bottom panel of each plot. 3$\sigma$ upper-limits are shown in grey. (*fig:seds_others*)

</div>
<div id="div_fig3">

<img src="tmp_2611.07034/./sources_f444_f1550.png" alt="Fig10" width="100%"/>

**Figure 10. -** Subtracted images for the NIRCam F444W (top) and MIRI F1550C (bottom) filters. The images are oriented with North pointing up and East pointing left. All candidate companions pre-vetting are marked with white markers. Not all sources were detected in all filters, so some markers appear empty.  (*fig:sources_f444_f1550*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.07034"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\jwst}{{\it JWST}}$
$\newcommand{\swift}{{\it Swift}}$
$\newcommand{\lacc}{L_{acc} }$
$\newcommand{\mic}{\mum }$</div>



<div id="title">

# Coordinated Space and Ground Monitoring of Accretion Bursts in a Protoplanetary Disk: \ MIRI and iSHELL observe the long-term modulation of gas within a small inner disk cavity

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.07253-b31b1b.svg)](https://arxiv.org/abs/2611.07253)<mark>Appeared on: 2026-11-06</mark> -  _Accepted for publication on ApJ_

</div>
<div id="authors">

J. Hyden, et al. -- incl., <mark>M. Benisty</mark>

</div>
<div id="abstract">

**Abstract:** DQ Tau is a TTauri eccentric binary with pulsed accretion bursts regulated by its $\sim 16$ -day orbital period. We have monitored this system with seven instruments from ground and space over 40 days. We present results on molecular gas variability from nine JWST-MIRI and nine IRTF-iSHELL epochs over three sequential accretion bursts with accretion luminosity pulses increasing by factors of $\sim 9$ --80. A fundamental new measurement from this analysis is the line broadening of CO (from iSHELL) and $\ce{H2O}$ (from MIRI) lines, which we use to estimate Keplerian radii and their variability in parallel to the line luminosity. This unprecedented dataset clearly shows that the molecular emission does not react like HI and the 5- $\mu$ m continuum with each accretion burst: the line luminosity and width of all hot molecular gas tracers decrease over $\sim30$ --35 days, consistent with an emitting region decreasing in temperature and mass as it recedes to larger radii within the $\sim 0.2$ --0.4 au inner cavity perturbed by the binary. The infrared molecular spectrum of DQ Tau is affected more by disk dynamics then by chemical changes triggered by variable accretion, supporting model expectations for chemical relaxation timescales that are much longer than the duration of these accretion bursts ( $\gg 10$ days). The variability observed in all tracers in this study is consistent with a scenario where gas mass builds up near the disk edge to modulate the burst peak intensity and gas density in the inner cavity every $\sim4$ --5 orbital periods. The origin of this modulation is still unclear.

</div>

<div id="div_fig1">

<img src="tmp_2611.07253/./Gas_Variability_Figure2_v9_15_7_new.png" alt="Fig1" width="100%"/>

**Figure 1. -**  Light curves of the HI-derived accretion luminosity (Appendix \ref{app: updated_Lacc_relations}), 5$\mu$m and 13$\mu$m dust continuum flux, and different molecular lines tracing gas at different temperatures \citep[from hotter gas with the 6000 K lines to colder gas with the 1500 K lines, see][]{banz25}. The vertical dashed lines indicate the three periastron orbital phases. The accretion luminosity light curve measured with LCO \cite[from][]{Tofflemire25} is reported in grey using the second y axis to the right. The colored (red to blue) markers indicate the JWST-MIRI data. The magenta points in the 5 $\mu$m continuum panel show the fluxes measured in iSHELL spectra and reported in Table \ref{tab: miri_epochs}. (*fig: line_var*)

</div>
<div id="div_fig2">

<img src="tmp_2611.07253/./illustration_lowerpanel_Final_15_7.png" alt="Fig16" width="100%"/>

**Figure 16. -** Overview of our proposed interpretation to reconcile all the observables with a longer-term modulation of molecular gas near the inner disk edge during this monitoring campaign. The top panel displays the full $\lacc$ time series collected in our program and in \citet{kospal25}, along with a sinusoidal curve to mark the tentative longer-term periodicity ($\sim 5 P_{bin}$) in the burst peak amplitude discussed in the text. The middle and lower panels show, as representative examples from the analysis in this work, the 6000 K water luminosity and the Keplerian radii for the FW20\% CO and 3500 K water lines, including for the later epochs from \citet{kospal25}. The barycenter distance of the primary is shown in the bottom plot for reference to the gas emission radii, illustrating how the high-velocity CO gas is temporarily pulled inward at the second and third periastron. (*fig: disc+cartoon*)

</div>
<div id="div_fig3">

<img src="tmp_2611.07253/./Slab_results_+radii.png" alt="Fig15" width="100%"/>

**Figure 15. -** Slab model fit results (from Figure \ref{fig: BestTNRM}) for the hot component (top) and warm component (middle) of water, shown as a function of the Keplerian radii independently measured from the line broadening (from Figure \ref{fig: Kepl radii}). The dashed curves in the temperature plots show power-law profiles with indices of -0.13 (hot component) and -0.3 (warm component), while the area plots show the expected trends from disk annuli that shrink (hot component) or expand with time (warm component), see text for details. The additional four epochs taken in 2026 (labeled K1 through 4) and published in \citet{kospal25} are included for comparison and agree with the values and trends found in our program. The plot in the bottom shows the hot component results as a function of $\lacc$; contrary to what reported in \citet{kospal25} based on 4 epochs, there is no correlation between the water temperature and $\lacc$. (*fig: Slab_results_+radii*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.07253"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand\mtrx{#1}$
$\newcommand\vctr{#1}$</div>



<div id="title">

# Mitigating scanning-induced systematics in _Hipparcos_ and implications for _Gaia_ astrometry

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.08282-b31b1b.svg)](https://arxiv.org/abs/2611.08282)<mark>Appeared on: 2026-11-06</mark> -  _6 pages, 5 figures. Accepted for publication in Astronomy & Astrophysics_

</div>
<div id="authors">

A. Binnenfeld, et al. -- incl., <mark>S. Shahaf</mark>, <mark>H.-W. Rix</mark>

</div>
<div id="abstract">

**Abstract:** Space-based astrometric missions such as _Hipparcos_ and _Gaia_ rely on one-dimensional scanning strategies that couple the measured signals with both observation times and scanning geometry. This coupling can introduce spurious periodicities and complicate their interpretation. We present a non-parametric extension of the phase-distance correlation (PDC) periodogram that quantifies and reduces dependence on scan geometry at the period-search stage. Using _Hipparcos_ data, we identified recurrent periodicities associated with characteristic scanning-law timescales and showed that they are strongly suppressed by the scan-angle-controlled statistic. We then applied the method to _Gaia_ Data Release 4 (DR4) prerelease epoch astrometry, where a recurrent $\sim$ 54-day feature is suppressed across multiple sources while independently known orbital signals remain detectable. The method provides a diagnostic of scan-angle dependence and a way to mitigate its impact on period searches. These results are particularly relevant for _Gaia_ DR4, where public epoch astrometry will enable large-scale searches for companions ranging from exoplanets to dormant black holes.

</div>

<div id="div_fig1">

<img src="tmp_2611.08282/./plots/ast_part_dist.png" alt="Fig1" width="100%"/>

**Figure 1. -** Distribution of $\mathcal{D}_{\mathrm{cpl}}$ for all _Hipparcos_ sources. The catalog has a mean of 0.09 and standard deviation of 0.15, with an evident tail indicating a subset of sources showing strong dependence on scan angle. (*fig:ast_part_dist*)

</div>
<div id="div_fig2">

<img src="tmp_2611.08282/./plots/ast_part_example1.png" alt="Fig2" width="100%"/>

**Figure 2. -** Standard (top) and partial (bottom) PDC periodograms for HIP 29616. A prominent peak at $f=0.0204 {\rm d}^{-1}$, corresponding to $P\simeq49.1$ days, is clearly visible in the standard PDC, consistent with a repeating signal driven by scan-angle systematics. This feature is effectively suppressed in the partial PDC. (*fig:ast_part_example1*)

</div>
<div id="div_fig3">

<img src="tmp_2611.08282/./plots/29616_orbit.png" alt="Fig3" width="100%"/>

**Figure 3. -** 
Best-fit astrometric orbit for HIP 29616 at the spurious $P\simeq49$ d period. The blue curve shows the best-fitting solution, and the along-scan measurements are overplotted. The faint curves show 80 realizations obtained by varying the fitted linear orbit coefficients within their estimated uncertainties, while keeping $P$, $e$, and $\tau$ fixed. The best-fitting solution has $P=49.243$ d, reaches the imposed eccentricity limit at $e=0.950$, and has an RMS residual of $6.496$ mas.
 (*fig:data29616_1*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.08282"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

137  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

12  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

6  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
